# Tutorial 3 — Cross-lingual transfer: English → Indonesian clickbait

**Question:** if we only have labeled clickbait data in English, can we detect clickbait in Indonesian?

We train on English headlines using **multilingual** sentence embeddings and test on the Indonesian CLICK-ID dataset, then try the cheapest possible fix: **re-tuning the decision threshold** with a few Indonesian labels.

**Runtime:** CPU is fine; GPU makes encoding faster.

In [ ]:
!pip -q install -U sentence-transformers datasets

## 1. Load both datasets

In [ ]:
from datasets import load_dataset
import pandas as pd

# English clickbait headlines (Chakraborty et al., 2016) — ~32k headlines, MIT licence
en = load_dataset("marksverdhei/clickbait_title_classification", split="train").to_pandas()
en = en.rename(columns={"title": "text", "clickbait": "label"})[["text", "label"]]
en = en.dropna().drop_duplicates("text").reset_index(drop=True)
print(en.shape)
print(en["label"].value_counts().rename({0: "not clickbait", 1: "clickbait"}))


In [ ]:
# Indonesian CLICK-ID headlines (William & Sari, 2020) — MTEB copy, 2,048 headlines, CC-BY-4.0
idd = load_dataset("mteb/IndonesianIdClickbaitClassification", split="train").to_pandas()[["text", "label"]]
print(idd.shape)
print(idd["label"].value_counts())

# Check the label convention by reading examples
for lab in sorted(idd.label.unique()):
    print(f"--- label {lab} ---")
    for t in idd[idd.label == lab].text.sample(4, random_state=0):
        print("  ", t)

In [ ]:
ID_CLICKBAIT = 1   # change this if the examples above show the opposite convention
idd["y"] = (idd["label"] == ID_CLICKBAIT).astype(int)

## 2. Encode both languages with the same multilingual encoder

In [ ]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
X_en = encoder.encode(en["text"].tolist(), batch_size=128, show_progress_bar=True, normalize_embeddings=True)
X_id = encoder.encode(idd["text"].tolist(), batch_size=128, show_progress_bar=True, normalize_embeddings=True)
y_en, y_id = en["label"].values, idd["y"].values
X_en.shape, X_id.shape

## 3. Split the Indonesian data

We pretend we can afford to label a **small calibration set** (30%) in Indonesian. Everything is evaluated on the remaining 70%, which is never used for any decision.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

idx_cal, idx_test = train_test_split(np.arange(len(idd)), test_size=0.7, stratify=y_id, random_state=42)
Xc, yc = X_id[idx_cal], y_id[idx_cal]
Xt, yt = X_id[idx_test], y_id[idx_test]
print("calibration:", len(idx_cal), " test:", len(idx_test))

## 4. Four strategies, one test set

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score

def evaluate(name, p, thr=0.5):
    pred = (p >= thr).astype(int)
    return {"strategy": name, "threshold": round(thr, 3),
            "precision": precision_score(yt, pred, zero_division=0),
            "recall": recall_score(yt, pred), "f1_macro": f1_score(yt, pred, average="macro"),
            "roc_auc": roc_auc_score(yt, p)}

def best_threshold(p, y):
    grid = np.linspace(0.01, 0.99, 99)
    return grid[np.argmax([f1_score(y, (p >= t).astype(int), average="macro") for t in grid])]

rows = []

# (a) English-only model, default threshold
en_clf = LogisticRegression(max_iter=2000).fit(X_en, y_en)
p_t = en_clf.predict_proba(Xt)[:, 1]
rows.append(evaluate("(a) EN model, zero-shot, t=0.5", p_t))

# (b) English-only model, threshold tuned on the Indonesian calibration set
t_star = best_threshold(en_clf.predict_proba(Xc)[:, 1], yc)
rows.append(evaluate("(b) EN model, threshold re-tuned", p_t, t_star))

# (c) Train only on the small Indonesian calibration set
id_clf = LogisticRegression(max_iter=2000).fit(Xc, yc)
rows.append(evaluate("(c) ID calibration set only", id_clf.predict_proba(Xt)[:, 1]))

# (d) English + Indonesian calibration set together
mix_clf = LogisticRegression(max_iter=2000).fit(np.vstack([X_en, Xc]), np.concatenate([y_en, yc]))
rows.append(evaluate("(d) EN + ID calibration set", mix_clf.predict_proba(Xt)[:, 1]))

pd.DataFrame(rows).round(4)

**Discuss:**
- In (a), compare precision and recall. Is the problem that the model *cannot tell* clickbait apart in Indonesian (low ROC-AUC), or that its scores are *shifted* (good ROC-AUC, bad recall)?
- Note that (a) and (b) have the **same ROC-AUC** — changing a threshold never changes the ranking. Why can it still change F1 so much?
- How close does (b) — one number tuned on a few hundred labels — get to (c) and (d)?

## 5. See the score shift

In [ ]:
import matplotlib.pyplot as plt

p_en = en_clf.predict_proba(X_en)[:, 1]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
for ax, p, y, title in [(axes[0], p_en, y_en, "English (training language)"),
                        (axes[1], p_t, yt, "Indonesian (zero-shot)")]:
    ax.hist(p[y == 0], bins=40, alpha=0.6, label="not clickbait", density=True)
    ax.hist(p[y == 1], bins=40, alpha=0.6, label="clickbait", density=True)
    ax.axvline(0.5, color="k", ls="--", lw=1, label="t = 0.5")
    ax.set_title(title); ax.set_xlabel("predicted P(clickbait)")
axes[1].axvline(t_star, color="red", lw=1.5, label=f"re-tuned t = {t_star:.2f}")
axes[0].legend(); axes[1].legend()
plt.tight_layout(); plt.show()

## Exercises

1. Shrink the calibration set to 5%, 10% and 20%. How few Indonesian labels do you need for (b) to work?
2. Try another multilingual encoder, e.g. `sentence-transformers/LaBSE` or `intfloat/multilingual-e5-base` (for e5, prefix each text with `"query: "`).
3. Which Indonesian words are most associated with clickbait? Fit a TF-IDF model on the Indonesian data and list the top coefficients. Are they topical or stylistic?
4. Collect 50 Polish headlines, label them yourself, and repeat the experiment for English → Polish.

---
*Data:* Chakraborty et al. (2016), *Stop Clickbait*, ASONAM · William, A. & Sari, Y. (2020), *CLICK-ID: A novel dataset for Indonesian clickbait headlines*, Data in Brief 32, 106231 (MTEB copy, CC-BY-4.0).